# Exercise 3 · Measure position bias in base Qwen3

**Part 04 · Local decision models** · 10 minutes

Before training anything, measure the base model with the same letter scorer we use later:

1. Score every test case with the options in the **given** order and in **reversed** order.
2. Compute how often the winner changes (the **flip rate**) and the accuracy of each order.
3. Implement **averaged** scoring over both orders.
4. Save the base model's results as the baseline for Demo 5.

Runs on CPU in a few minutes for Qwen3-0.6B; faster on a GPU.

In [ ]:
%pip install -q -r requirements-finetune.txt

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json
from dmw import scorer as S, datagen, metrics, checks
from dmw.prompt import build_messages

BASE_MODEL = "Qwen/Qwen3-0.6B"      # the model we fine-tune in notebook 07
model, tok = S.load(BASE_MODEL)
sc = S.LetterScorer(model, tok)
cases = datagen.load_test("data/fresh_test.json")
print(len(cases), "test cases on", config.device())

In [ ]:
# What the model actually sees for one case
c = cases[0]
print(S.render_prompt(tok, build_messages(c["context"], list(c["choices"]), None, c["question"], c["choices"]))[-900:])

## 1 · Score one order at a time

In [ ]:
def winner(case, order):
    return sc.score(case["context"], list(case["choices"]), question=case["question"],
                    descriptions=case["choices"], orders=[order]).choice

given_pred, reversed_pred, p_first = [], [], []
for c in cases:
    order = list(c["choices"])
    d = sc.score(c["context"], order, question=c["question"], descriptions=c["choices"], orders=[order])
    given_pred.append(d.choice); p_first.append(d.per_order[0]["probabilities"][order[0]])
    reversed_pred.append(winner(c, order[::-1]))

## 2 · Flip rate and accuracy per order

In [ ]:
gold = [c["expected"] for c in cases]
flip_rate = sum(a != b for a, b in zip(given_pred, reversed_pred)) / len(cases)
given_acc = metrics.accuracy(gold, given_pred)
reversed_acc = metrics.accuracy(gold, reversed_pred)
print(f"flip rate {flip_rate:.2%} | given {given_acc:.2%} | reversed {reversed_acc:.2%} | "
      f"mean P(first option) {sum(p_first)/len(p_first):.2f} (unbiased ≈ 1/n)")

## 3 · Averaged scoring

In [ ]:
def averaged(case):
    order = list(case["choices"])
    return sc.score(case["context"], order, question=case["question"], descriptions=case["choices"],
                    orders=[order, order[::-1]]).choice

averaged_acc = metrics.accuracy(gold, [averaged(c) for c in cases])
print(f"averaged {averaged_acc:.2%}")

In [ ]:
reference = checks.position_bias_reference(sc, cases)
checks.check_position_bias(flip_rate, given_acc, reversed_acc, averaged_acc, reference)

## 4 · Save the baseline for Demo 5 (sorted + reversed orderings, like Stavro)

In [ ]:
rows = S.score_cases(sc, cases)
summary = metrics.summarize(rows)
json.dump({"model": BASE_MODEL, "rows": rows, "summary": summary},
          open(config.OUTPUTS / "baseline_base_model.json", "w"), indent=1)
print("letter mass (share of probability on the option letters):",
      round(sum(r["letter_mass"] for r in rows) / len(rows), 3))
metrics.summary_table(summary)

### Discussion
Averaging is not guaranteed to beat the better single order on every set, but it removes the
dependence on an arbitrary choice of order, and the disagreement between orders is a free
signal for review (`orderings_disagree`). A base model that flips often is guessing by position.